In [ ]:
%pip install -r requirements.txt
%pip install ipykernel

Then in a bash terminal, run the command "build_freeciv_web_service" . This command will start the docker container needed for running civrealm.

This cell must be run in order to stop Jupyter's running loop from crashing our code.

In [ ]:
import os
os.environ["RAY_DEDUP_LOGS"] = "0"

import nest_asyncio
nest_asyncio.apply()


In [ ]:
import time
import math
import random
from collections import deque
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import civrealm
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


This function is used to create our environment using gymnnasium. This is needed as we will often create new envs.

In [ ]:
def make_env_fn(seed: int = 0):
    env = gym.make(
        "civrealm/FreecivTensorMinitask-v0",
        minitask_pattern={"type": "battle_ancient_era"},
    )
    # reset here so the env is guaranteed ready before returning
    env.reset(seed=seed)
    return env
env = make_env_fn()


# DQN


## Observation Encoder

This module converts CivRealm’s heterogeneous observation dictionary into a
single fixed-size latent vector suitable for deep reinforcement learning.

CivRealm observations contain:
- Spatial grids (`map`)
- Variable-length entity lists (`unit`, `city`)
- Global vectors (`player`, `rules`)

Each modality is processed with a network appropriate to its structure:
- **CNN** for spatial map data
- **MLPs with mean pooling** for variable-size entity sets
- **MLPs** for global features

Mean pooling ensures permutation invariance and stability when the number of
units or cities changes over time.

All modality embeddings are concatenated and projected into a single latent
state vector, which is then used by the Q-network (or policy/value networks).

This design mirrors the representation strategy used in the CivRealm paper and
is essential for stable learning in large, structured strategy environments.


In [ ]:
class ObsEncoder(nn.Module):
    """
    Encodes a CivRealm tensor-dict observation into a single fixed-size
    feature vector suitable for deep RL (DQN / Branching DQN / PPO).

    Modalities used:
      - map    : spatial grid (16 x 16 x 112)
      - unit   : variable number of units (Nunit x 125)
      - city   : variable number of cities (Ncity x 248)
      - player : global player stats (113)
      - rules  : game rules and constants (120)
    """

    def __init__(self, out_dim: int = 512):
        super().__init__()
        self.out_dim = out_dim

        # ------------------------------------------------------------
        # MAP ENCODER (CNN)
        # ------------------------------------------------------------
        # The map is a spatial grid with many symbolic channels.
        # A CNN is used to extract spatial and strategic patterns
        # such as terrain layout, unit positions, and city locations.
        #
        # Input shape : (B, 16, 16, 112)
        # Output      : (B, 256)
        self.map_cnn = nn.Sequential(
            # Learn local spatial features while preserving resolution
            nn.Conv2d(112, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Downsample to capture larger-scale spatial structure
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1),  # 16 -> 8
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # 8 -> 4
            nn.ReLU(),

            # Pool spatial dimensions to produce a fixed-size vector
            nn.AdaptiveAvgPool2d((1, 1)),  # (B, 256, 1, 1)
            nn.Flatten(),                  # (B, 256)
        )

        # ------------------------------------------------------------
        # UNIT ENCODER (MLP + MEAN POOLING)
        # ------------------------------------------------------------
        # Units are variable in number and unordered.
        # Mean pooling makes the representation permutation-invariant
        # and robust to changing unit counts.
        self.unit_mlp = nn.Sequential(
            nn.Linear(125, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )

        # ------------------------------------------------------------
        # CITY ENCODER (MLP + MEAN POOLING)
        # ------------------------------------------------------------
        # Same logic as units: variable count, unordered entities.
        self.city_mlp = nn.Sequential(
            nn.Linear(248, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )

        # ------------------------------------------------------------
        # PLAYER ENCODER (MLP)
        # ------------------------------------------------------------
        # Player features are already global and fixed-size.
        self.player_mlp = nn.Sequential(
            nn.Linear(113, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )

        # ------------------------------------------------------------
        # RULES ENCODER (MLP)
        # ------------------------------------------------------------
        # Game rules affect what actions are optimal.
        # Encoding them helps generalization across scenarios.
        self.rules_mlp = nn.Sequential(
            nn.Linear(120, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
        )

        # ------------------------------------------------------------
        # FUSION LAYER
        # ------------------------------------------------------------
        # Concatenates all modality embeddings into a single
        # latent state representation.
        fused_in = 256 + 256 + 256 + 128 + 128
        self.fuse = nn.Sequential(
            nn.Linear(fused_in, out_dim),
            nn.ReLU(),
        )

    def forward(self, obs: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        Forward pass that converts a CivRealm observation dictionary
        into a single latent feature vector.

        Returns:
            Tensor of shape (B, out_dim)
        """

        # Convert integer observations to float for neural networks
        map_t    = obs["map"].float()
        unit_t   = obs["unit"].float()
        city_t   = obs["city"].float()
        player_t = obs["player"].float()
        rules_t  = obs["rules"].float()

        # Add batch dimension if a single observation is provided
        if map_t.dim() == 3:
            map_t    = map_t.unsqueeze(0)
            unit_t   = unit_t.unsqueeze(0)
            city_t   = city_t.unsqueeze(0)
            player_t = player_t.unsqueeze(0)
            rules_t  = rules_t.unsqueeze(0)

        # Reorder map tensor to channel-first format for PyTorch CNNs
        # (B, 16, 16, 112) -> (B, 112, 16, 16)
        map_t = map_t.permute(0, 3, 1, 2)

        # Encode each modality
        map_feat    = self.map_cnn(map_t)                 # (B, 256)
        unit_feat   = self.unit_mlp(unit_t.mean(dim=1))   # (B, 256)
        city_feat   = self.city_mlp(city_t.mean(dim=1))   # (B, 256)
        player_feat = self.player_mlp(player_t)           # (B, 128)
        rules_feat  = self.rules_mlp(rules_t)             # (B, 128)

        # Concatenate all features and project to final latent space
        fused = torch.cat(
            [map_feat, unit_feat, city_feat, player_feat, rules_feat],
            dim=-1
        )

        return self.fuse(fused)  # (B, out_dim)


## Branching Q-Network (Factorized DQN)


CivRealm’s action space is not a single discrete variable but a structured
dictionary of discrete decisions (e.g., actor type, entity ID, action type).

Flattening this into one huge action space would be infeasible and unstable.
Instead, we use a **Branching (Factorized) DQN** architecture:

- A **shared observation encoder** produces a latent state representation
- A **separate Q-head** is learned for each action branch
- Each head outputs Q-values only for its corresponding decision dimension

This allows the agent to:
- Scale to very large structured action spaces
- Learn reusable state features across decisions
- Select actions by combining per-branch argmax operations

This design follows the Branching DQN principle (Tavakoli et al., 2018) and is
well-suited to CivRealm’s tensor-based API.


In [ ]:
import torch
import torch.nn as nn
from typing import Dict
from gymnasium import spaces

class BranchingQNet(nn.Module):
    """
    Branching (Factorized) Q-Network for structured action spaces.

    Instead of producing a single flat Q-vector, this network:
      - Uses a shared observation encoder
      - Outputs one Q-value vector per action branch

    This is essential for environments like CivRealm where actions
    are represented as a dictionary of discrete decisions:
      - actor_type
      - unit_id
      - unit_action_type
      - city_id
      - city_action_type
      - etc.

    Forward output:
        Dict[str, Tensor], where each tensor has shape (B, n_branch)
    """

    def __init__(
        self,
        obs_space: spaces.Dict,
        action_space: spaces.Dict,
        feat_dim: int = 512
    ):
        super().__init__()

        # ------------------------------------------------------------
        # SHARED OBSERVATION ENCODER
        # ------------------------------------------------------------
        # Converts the CivRealm observation dictionary into a single
        # latent feature vector z ∈ R^{feat_dim}
        self.encoder = ObsEncoder(out_dim=feat_dim)

        # ------------------------------------------------------------
        # ACTION BRANCHES
        # ------------------------------------------------------------
        # CivRealm actions are a Dict of Discrete spaces.
        # Each key corresponds to an independent decision dimension.
        assert isinstance(action_space, spaces.Dict)

        # Keep branch names in a fixed order
        self.branches = list(action_space.spaces.keys())

        # Create one Linear "head" per action branch
        # Each head outputs Q-values for that branch only
        heads = {}
        for branch_name, branch_space in action_space.spaces.items():
            if not isinstance(branch_space, spaces.Discrete):
                raise ValueError(
                    f"Branch {branch_name} is not Discrete: {branch_space}"
                )

            # One Q-vector per branch:
            # Q_branch(s, ·) ∈ R^{n_actions_in_branch}
            heads[branch_name] = nn.Linear(feat_dim, branch_space.n)

        # ModuleDict ensures parameters are properly registered
        self.heads = nn.ModuleDict(heads)

    def forward(self, obs: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """
        Forward pass.

        Args:
            obs: CivRealm observation dictionary

        Returns:
            Dictionary mapping each action branch to its Q-values:
              {
                branch_name: Tensor of shape (B, n_branch_actions)
              }
        """

        # Encode observation into shared latent representation
        z = self.encoder(obs)  # (B, feat_dim)

        # Compute Q-values independently for each action branch
        q_values = {}
        for branch_name, head in self.heads.items():
            q_values[branch_name] = head(z)  # (B, n_branch_actions)

        return q_values


# ------------------------------------------------------------
# QUICK SANITY CHECK
# ------------------------------------------------------------
# Ensures the network builds correctly and can run a forward pass
qnet_test = BranchingQNet(
    env.observation_space,
    env.action_space
).to(DEVICE)


## Utilities


### Observation Packing and Action Mask Utilities



This cell provides utility functions that bridge CivRealm’s environment API
with a Branching DQN agent.

#### Why this code is needed

CivRealm:
- Returns observations as large dictionaries with many keys
- Provides action validity through masks
- Mixes NumPy arrays and torch tensors

Deep RL models:
- Expect only the features they were trained on
- Require tensors on the correct device
- Must never select invalid actions

#### What this cell does

1. **Observation packing**
   - Selects only the keys used by the encoder
   - Converts everything to torch tensors
   - Moves data to CPU/GPU consistently

2. **Mask extraction**
   - Handles both CivRealm mask formats
   - Produces a unified dictionary of NumPy masks

3. **Masked action selection**
   - `masked_argmax` chooses the best valid action
   - `sample_from_mask` enables safe random exploration

These utilities are essential to prevent invalid actions,
environment crashes, and unstable training.


In [ ]:
import numpy as np
import torch
from typing import Dict, Any

# ------------------------------------------------------------
# OBSERVATION KEYS USED BY THE ENCODER
# ------------------------------------------------------------
# We explicitly restrict the observation to the modalities
# consumed by ObsEncoder. This avoids passing unused tensors
# and keeps replay buffer entries small and stable.
ENC_KEYS = ("map", "unit", "city", "player", "rules")


def pack_obs_for_encoder(
    obs: Dict[str, Any],
    device: torch.device
) -> Dict[str, torch.Tensor]:
    """
    Prepare a CivRealm observation dictionary for the neural encoder.

    - Keeps only the keys required by ObsEncoder
    - Converts everything to torch.Tensor
    - Moves tensors to the correct device (CPU/GPU)

    This function ensures a clean interface between the environment
    (NumPy-based) and the PyTorch model.
    """
    out = {}
    for k in ENC_KEYS:
        v = obs[k]

        # If already a tensor, just move it to device
        if isinstance(v, torch.Tensor):
            t = v.to(device)
        else:
            # Convert NumPy / list / scalar to torch.Tensor
            t = torch.as_tensor(v, device=device)

        out[k] = t

    return out


def _to_np(x: Any) -> np.ndarray:
    """
    Convert input to a NumPy array safely.

    Used when extracting masks, since action selection
    and masking logic is done in NumPy (outside autograd).
    """
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def extract_masks(obs: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """
    Extract action masks from a CivRealm observation.

    CivRealm may expose masks in two different ways:
      1. obs["action_mask"] as a dictionary
      2. Individual top-level keys ending with "_mask"

    This function unifies both formats into a single dictionary:
        mask_name -> NumPy array
    """
    masks: Dict[str, np.ndarray] = {}

    # Case 1: grouped action_mask dictionary
    if (
        isinstance(obs, dict)
        and "action_mask" in obs
        and isinstance(obs["action_mask"], dict)
    ):
        for k, v in obs["action_mask"].items():
            masks[k] = _to_np(v)

    # Case 2: top-level *_mask keys
    if isinstance(obs, dict):
        for k, v in obs.items():
            if isinstance(k, str) and k.endswith("_mask"):
                masks[k] = _to_np(v)

    return masks


def masked_argmax(values_1d: np.ndarray, mask_1d: np.ndarray) -> int:
    """
    Compute argmax over Q-values while respecting an action mask.

    Invalid actions (mask == 0 / False) are ignored.

    If:
      - no valid actions exist, OR
      - inputs are empty
    the function safely returns action index 0.
    """
    v = np.asarray(values_1d).reshape(-1)
    m = np.asarray(mask_1d).reshape(-1).astype(bool)

    # Safety checks
    if m.size == 0 or v.size == 0:
        return 0

    # Align lengths defensively
    n = min(len(v), len(m))
    v = v[:n]
    m = m[:n]

    # If no valid action is available
    if not m.any():
        return 0

    # Mask invalid actions by setting them to -infinity
    vv = v.copy()
    vv[~m] = -1e30

    return int(np.argmax(vv))


def sample_from_mask(mask_1d: np.ndarray) -> int:
    """
    Uniformly sample a valid action index from a binary mask.

    Used for epsilon-greedy exploration when selecting random actions.

    If no valid actions exist, returns 0 safely.
    """
    m = np.asarray(mask_1d).reshape(-1).astype(bool)

    if m.size == 0 or not m.any():
        return 0

    idxs = np.where(m)[0]
    return int(np.random.choice(idxs))


### Masked Epsilon-Greedy Action Selection for Branching DQN


CivRealm uses a dictionary action space with multiple discrete branches (e.g., `unit_id`, `unit_action_type`, `city_id`, etc.).
Not all actions are valid at every step, so CivRealm provides **action masks** in the observation.

This function implements **epsilon-greedy exploration** while *always respecting masks*:

#### Random (epsilon) branch
- Chooses random actions **only among valid actions** using the mask.
- Handles hierarchical dependencies:
  - pick `unit_id` first
  - then use `unit_action_type_mask[unit_id]` to pick a valid unit action

#### Greedy (1 - epsilon) branch
- Runs the Q-network to get Q-values for each branch.
- Uses `masked_argmax` so invalid actions are never selected.

#### Why this is essential
Without masked selection:
- epsilon exploration will frequently choose illegal actions
- training becomes unstable or crashes
- the agent “learns” from inconsistent transitions


In [ ]:
def select_action_branching(
    env,
    qnet: "BranchingQNet",
    obs: Dict[str, Any],
    device: torch.device,
    epsilon: float,
) -> Dict[str, int]:
    """
    Select an action for CivRealm's *branching* (multi-discrete-as-dict) action space.

    Core requirements this function satisfies:
    1) Epsilon-greedy exploration (random with prob epsilon, greedy otherwise)
    2) ALWAYS respect action masks (CivRealm forbids invalid actions)
    3) Support hierarchical / conditional masks:
         - Some branches depend on an entity choice (e.g., unit_action_type depends on unit_id)
         - Masks may be 1D or 2D matrices (entity x action_type)
    """

    # CivRealm encodes which actions are legal via masks inside the observation.
    # We unify all possible mask formats into a flat dict: mask_name -> np.ndarray
    masks = extract_masks(obs)

    # ------------------------------------------------------------
    # Helper: get a 1D mask for a branch (default = all valid)
    # ------------------------------------------------------------
    def get_mask(branch: str, n: int) -> np.ndarray:
        """
        Return a length-n mask for a branch.
        - If mask is missing: assume all actions are valid (ones)
        - If mask is shorter than n: pad with zeros (invalid)
        - If mask is longer than n: truncate
        """
        key = f"{branch}_mask"
        if key in masks:
            m = np.asarray(masks[key]).reshape(-1)

            # If the env gave fewer entries than action_space.n,
            # we pad missing entries as invalid (0) to avoid out-of-bounds.
            if len(m) < n:
                m2 = np.zeros((n,), dtype=np.int32)
                m2[:len(m)] = m.astype(np.int32)
                return m2

            # If too long, truncate to match the branch size.
            return m[:n].astype(np.int32)

        # If no mask provided, assume everything is valid.
        return np.ones((n,), dtype=np.int32)

    # ------------------------------------------------------------
    # Helper: get a row from a 2D mask matrix (entity-conditioned)
    # ------------------------------------------------------------
    def get_row_mask(key: str, row: int, n: int) -> np.ndarray:
        """
        Some masks depend on a chosen entity:
          unit_action_type_mask shape ~ (num_units, num_unit_action_types)

        This returns the mask row corresponding to the selected entity index.
        - If missing: all actions valid
        - If mask is actually 1D (unexpected): fall back to get_mask
        - Clamp row to valid bounds
        - Pad/truncate to length n
        """
        if key not in masks:
            return np.ones((n,), dtype=np.int32)

        mat = np.asarray(masks[key])

        # Some environments might accidentally provide this as a 1D vector.
        if mat.ndim == 1:
            # Convert "unit_action_type_mask" -> "unit_action_type"
            return get_mask(key.replace("_mask", ""), n)

        # Clamp row index so we never index outside the matrix.
        row = int(np.clip(row, 0, mat.shape[0] - 1)) if mat.shape[0] > 0 else 0

        m = mat[row].reshape(-1)

        # Same padding/truncation logic as 1D masks.
        if len(m) < n:
            m2 = np.zeros((n,), dtype=np.int32)
            m2[:len(m)] = m.astype(np.int32)
            return m2

        return m[:n].astype(np.int32)

    # ------------------------------------------------------------
    # Initialize action dict with default zeros for every branch.
    # ------------------------------------------------------------
    # CivRealm expects a full dict with an int for each branch key.
    action = {k: 0 for k in env.action_space.spaces.keys()}

    # ============================================================
    # (A) EPSILON-RANDOM EXPLORATION (BUT STILL MASKED!)
    # ============================================================
    # Important: random exploration must still obey masks,
    # otherwise we will attempt illegal actions and crash / get rejected.
    if np.random.rand() < epsilon:

        # --- actor_type: some action spaces select a "mode" first (unit/city/dipl/etc.)
        if "actor_type" in env.action_space.spaces:
            n_actor = env.action_space["actor_type"].n
            actor_mask = get_mask("actor_type", n_actor)
            actor_type = sample_from_mask(actor_mask)
            action["actor_type"] = int(actor_type)

        # --- UNIT BRANCHES (entity choice + dependent action type)
        if "unit_id" in env.action_space.spaces:
            n_uid = env.action_space["unit_id"].n
            uid_mask = get_mask("unit_id", n_uid)
            uid = sample_from_mask(uid_mask)
            action["unit_id"] = int(uid)

            # unit_action_type is conditioned on which unit was selected
            if "unit_action_type" in env.action_space.spaces:
                n_uat = env.action_space["unit_action_type"].n
                uat_mask_row = get_row_mask("unit_action_type_mask", uid, n_uat)
                action["unit_action_type"] = int(sample_from_mask(uat_mask_row))

        # --- CITY BRANCHES
        if "city_id" in env.action_space.spaces:
            n_cid = env.action_space["city_id"].n
            cid_mask = get_mask("city_id", n_cid)
            cid = sample_from_mask(cid_mask)
            action["city_id"] = int(cid)

            if "city_action_type" in env.action_space.spaces:
                n_cat = env.action_space["city_action_type"].n
                cat_mask_row = get_row_mask("city_action_type_mask", cid, n_cat)
                action["city_action_type"] = int(sample_from_mask(cat_mask_row))

        # --- DIPLOMACY BRANCHES
        if "dipl_id" in env.action_space.spaces:
            n_did = env.action_space["dipl_id"].n
            did_mask = get_mask("dipl_id", n_did)
            did = sample_from_mask(did_mask)
            action["dipl_id"] = int(did)

            if "dipl_action_type" in env.action_space.spaces:
                n_dat = env.action_space["dipl_action_type"].n
                dat_mask_row = get_row_mask("dipl_action_type_mask", did, n_dat)
                action["dipl_action_type"] = int(sample_from_mask(dat_mask_row))

        # --- GOV / TECH: usually just single masked discrete branches
        for b in ["gov_action_type", "tech_action_type"]:
            if b in env.action_space.spaces:
                n = env.action_space[b].n
                action[b] = int(sample_from_mask(get_mask(b, n)))

        return action

    # ============================================================
    # (B) GREEDY EXPLOITATION (MASKED ARGMAX OVER Q-VALUES)
    # ============================================================
    # We compute Q-values for each branch, then take argmax among valid actions.
    qnet.eval()
    with torch.no_grad():
        # Convert env observation into the minimal tensor dict the encoder expects.
        obs_t = pack_obs_for_encoder(obs, device)

        # Forward pass gives a dict: branch -> Q-values (B, n_branch)
        q = qnet(obs_t)

    # Convenience: extract the 1D numpy Q vector for a branch (since B=1 during acting)
    def q1(branch: str) -> np.ndarray:
        return q[branch].squeeze(0).detach().cpu().numpy().reshape(-1)

    # --- actor_type (masked greedy)
    if "actor_type" in env.action_space.spaces:
        n_actor = env.action_space["actor_type"].n
        actor_mask = get_mask("actor_type", n_actor)
        actor_type = masked_argmax(q1("actor_type"), actor_mask)
        action["actor_type"] = int(actor_type)

    # --- unit_id (masked greedy)
    if "unit_id" in env.action_space.spaces:
        n_uid = env.action_space["unit_id"].n
        uid_mask = get_mask("unit_id", n_uid)
        uid = masked_argmax(q1("unit_id"), uid_mask)
        action["unit_id"] = int(uid)

        # unit_action_type depends on chosen unit_id, so use row mask for that uid
        if "unit_action_type" in env.action_space.spaces:
            n_uat = env.action_space["unit_action_type"].n
            uat_mask_row = get_row_mask("unit_action_type_mask", uid, n_uat)
            action["unit_action_type"] = int(
                masked_argmax(q1("unit_action_type"), uat_mask_row)
            )

    # --- city_id (masked greedy)
    if "city_id" in env.action_space.spaces:
        n_cid = env.action_space["city_id"].n
        cid_mask = get_mask("city_id", n_cid)
        cid = masked_argmax(q1("city_id"), cid_mask)
        action["city_id"] = int(cid)

        if "city_action_type" in env.action_space.spaces:
            n_cat = env.action_space["city_action_type"].n
            cat_mask_row = get_row_mask("city_action_type_mask", cid, n_cat)
            action["city_action_type"] = int(
                masked_argmax(q1("city_action_type"), cat_mask_row)
            )

    # --- dipl_id (masked greedy)
    if "dipl_id" in env.action_space.spaces:
        n_did = env.action_space["dipl_id"].n
        did_mask = get_mask("dipl_id", n_did)
        did = masked_argmax(q1("dipl_id"), did_mask)
        action["dipl_id"] = int(did)

        if "dipl_action_type" in env.action_space.spaces:
            n_dat = env.action_space["dipl_action_type"].n
            dat_mask_row = get_row_mask("dipl_action_type_mask", did, n_dat)
            action["dipl_action_type"] = int(
                masked_argmax(q1("dipl_action_type"), dat_mask_row)
            )

    # --- gov / tech (masked greedy)
    for b in ["gov_action_type", "tech_action_type"]:
        if b in env.action_space.spaces:
            n = env.action_space[b].n
            action[b] = int(masked_argmax(q1(b), get_mask(b, n)))

    return action


## Epsilon-greedy action selection with mask support


### Summing Q-values for Branching DQN Actions

In Branching (Factorized) DQN, the action is not a single discrete choice,
but a **dictionary of independent discrete decisions**:

\[
a = (a_1, a_2, \dots, a_K)
\]

Instead of learning a single Q-function over the Cartesian product of actions,
we approximate the joint action-value as:

\[
Q(s, a) \approx \sum_{k=1}^{K} Q_k(s, a_k)
\]

#### What this cell does

- `sum_q_for_action`:
  - Extracts the Q-value corresponding to each selected branch action
  - Sums them to form a scalar joint Q-value
  - This value is used in the TD loss

- `action_dict_to_torch`:
  - Converts environment actions (Python ints) into batched torch tensors
  - Ensures compatibility with PyTorch operations and GPU execution

#### Why this is essential

Without this logic:
- The TD target cannot be computed correctly
- The network would not know which per-branch Q-values to update
- Branching DQN would silently degenerate into an incorrect objective


In [ ]:
def sum_q_for_action(
    q_dict: Dict[str, torch.Tensor],
    action: Dict[str, torch.Tensor]
) -> torch.Tensor:
    """
    Compute the total Q-value for a *composite action* in Branching DQN.

    Inputs:
        q_dict:
            Dictionary mapping each action branch to its Q-values:
              branch -> Tensor of shape (B, n_branch_actions)

        action:
            Dictionary mapping each branch to the selected action index:
              branch -> Tensor of shape (B,) with dtype long

    Output:
        Tensor of shape (B,) representing:
            Q(s, a) = sum_k Q_k(s, a_k)

    This implements the Branching DQN approximation:
        Q(s, a) ≈ Σ_k Q_k(s, a_k)
    """

    total = None  # will accumulate per-branch Q-values

    for branch_name, q_branch in q_dict.items():
        # Skip branches that are not part of the action
        # (this allows flexibility if some branches are unused)
        if branch_name not in action:
            continue

        # Selected action indices for this branch
        # Shape: (B,)
        a_branch = action[branch_name].long()

        # Safety clamp:
        # Ensures the index is within [0, n_actions - 1]
        # Protects against rare edge cases or malformed replay entries
        a_branch = a_branch.clamp(min=0, max=q_branch.shape[1] - 1)

        # Gather Q-values for the selected action:
        # q_branch: (B, n)
        # gather -> (B, 1)
        picked_q = q_branch.gather(1, a_branch.view(-1, 1)).squeeze(1)

        # Accumulate across branches
        if total is None:
            total = picked_q
        else:
            total = total + picked_q

    return total


def action_dict_to_torch(
    action: Dict[str, int],
    device: torch.device
) -> Dict[str, torch.Tensor]:
    """
    Convert a Python action dictionary into a torch-compatible format.

    Input:
        action:
            branch -> int (single action index)

    Output:
        branch -> Tensor of shape (1,) on the correct device

    Why this is needed:
    - Environment actions are Python ints
    - Q-networks and loss functions expect torch tensors
    - We wrap each action in a batch dimension (B=1)
    """

    return {
        branch: torch.tensor([int(value)], device=device, dtype=torch.long)
        for branch, value in action.items()
    }


## Replay Buffer


This cell defines the data structures used to store and sample experience
for DQN training.

#### Transition

Each `Transition` corresponds to one environment step:

\[
(s_t, a_t, r_t, s_{t+1}, \text{done})
\]

Only the observation components used by the encoder are stored,
which keeps the replay buffer compact and efficient.

#### ReplayBuffer

The replay buffer:
- Stores transitions in a fixed-size FIFO queue
- Discards old experiences automatically
- Samples transitions uniformly at random

This breaks temporal correlations between consecutive steps,
which is essential for stable Q-learning.

#### strip_obs

CivRealm observations contain many fields (masks, metadata, debug info).
This helper function:
- Keeps only the encoder-relevant keys
- Converts everything to NumPy arrays
- Ensures replay buffer entries are device-independent


In [ ]:
@dataclass
class Transition:
    """
    One experience tuple stored in the replay buffer.

    This corresponds to a single environment step:
        (s, a, r, s', done)

    We store observations as NumPy arrays (not torch tensors)
    to keep the replay buffer lightweight and device-agnostic.
    """
    obs: Dict[str, np.ndarray]        # Current state (encoder keys only)
    next_obs: Dict[str, np.ndarray]   # Next state (encoder keys only)
    action: Dict[str, int]            # Branching action taken
    reward: float                     # Scalar reward
    done: bool                        # Episode termination flag


class ReplayBuffer:
    """
    Experience replay buffer for DQN-style algorithms.

    Stores transitions and allows uniform random sampling.
    """

    def __init__(self, capacity: int = 50_000):
        # Deque automatically discards old transitions when full
        self.buf = deque(maxlen=capacity)

    def push(self, t: Transition):
        """
        Add a transition to the buffer.

        Oldest transitions are discarded automatically
        when capacity is exceeded.
        """
        self.buf.append(t)

    def sample(self, batch_size: int) -> List[Transition]:
        """
        Uniformly sample a minibatch of transitions.

        Random sampling breaks temporal correlations,
        which is crucial for stable DQN learning.
        """
        return random.sample(self.buf, batch_size)

    def __len__(self):
        """
        Allows using len(buffer) to check
        whether enough samples exist to start learning.
        """
        return len(self.buf)


def strip_obs(obs: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """
    Extract only the observation components used by the encoder
    and convert them to NumPy arrays.

    Why this exists:
    - CivRealm observations contain many keys (masks, metadata, etc.)
    - The encoder only needs a fixed subset (ENC_KEYS)
    - Replay buffers should store compact, serializable data

    Returns:
        Dict[str, np.ndarray] suitable for ReplayBuffer storage
    """
    return {k: _to_np(obs[k]) for k in ENC_KEYS}


## DQN Loss for Branching DQN


### Branching DQN TD Loss (1-step)

This cell computes the training loss used to update the Branching DQN.

#### What we want to learn
We want the network to approximate the action-value function:

\[
Q(s,a) \approx \mathbb{E}\left[\sum_{t\ge0}\gamma^t r_{t}\right]
\]

Because the action is a **dict of discrete branches** (actor_type, unit_id, etc.),
we use the common factorized approximation:

\[
Q_{\text{total}}(s,a) \approx \sum_{k} Q_k(s,a_k)
\]

#### TD target
We use a one-step TD target computed with the **target network**:

\[
y = r + \gamma (1-\text{done})\max_{a'} Q_{\text{target}}(s',a')
\]

We approximate the max over the joint action by summing per-branch maxima:

\[
\max_{a'} Q(s',a') \approx \sum_k \max_{a_k'} Q_k(s',a_k')
\]

#### Why Huber loss
We use Smooth L1 (Huber) loss instead of MSE because it is more robust
to occasional large TD errors, which helps stabilize DQN training.


In [ ]:
def compute_branching_dqn_loss(
    qnet: BranchingQNet,
    qnet_target: BranchingQNet,
    batch: List[Transition],
    env,
    gamma: float,
    device: torch.device,
) -> torch.Tensor:
    """
    Compute the 1-step TD loss for a Branching DQN.

    Key idea:
      - The action is *multi-branch* (a dict of discrete choices).
      - We approximate Q(s,a) by SUMMING the per-branch Q-values:
            Q_total(s, a_dict) = Σ_k Q_k(s, a_k)

    TD target (one-step):
        y = r + γ (1 - done) * max_{a'} Q_target(s', a')

    Here we approximate the max over joint actions by summing independent
    per-branch maxima:
        max_{a'} Q_target(s', a')  ≈  Σ_k max_{a_k'} Q_target,k(s', a_k')

    This approximation is common in branching / factorized-action setups.
    """

    # ------------------------------------------------------------
    # 1) Convert the replay batch (list of Transition objects)
    #    into batched torch tensors on the correct device.
    # ------------------------------------------------------------
    B = len(batch)  # batch size

    # Stack observations into (B, ...) tensors for each encoder key.
    # We stored obs as numpy arrays in the ReplayBuffer, so we stack
    # along axis=0 to create a minibatch.
    obs_batch = {
        k: torch.as_tensor(
            np.stack([t.obs[k] for t in batch], axis=0),
            device=device
        )
        for k in ENC_KEYS
    }

    next_obs_batch = {
        k: torch.as_tensor(
            np.stack([t.next_obs[k] for t in batch], axis=0),
            device=device
        )
        for k in ENC_KEYS
    }

    # Rewards and dones become (B,) float tensors
    rewards = torch.as_tensor(
        [t.reward for t in batch],
        device=device,
        dtype=torch.float32
    )
    dones = torch.as_tensor(
        [t.done for t in batch],
        device=device,
        dtype=torch.float32
    )

    # ------------------------------------------------------------
    # 2) Compute current Q(s,a) under the ONLINE network.
    # ------------------------------------------------------------
    # qnet(obs_batch) returns:
    #   dict branch -> tensor of shape (B, n_actions_for_branch)
    q_dict = qnet(obs_batch)

    # Build an action tensor for each branch: shape (B,)
    # If a branch is missing in a transition (shouldn't happen often),
    # we safely default to action 0.
    act_t = {}
    for k in env.action_space.spaces.keys():
        act_t[k] = torch.as_tensor(
            [t.action.get(k, 0) for t in batch],
            device=device,
            dtype=torch.long
        )

    # Pick Q-values corresponding to the taken actions and SUM across branches:
    # q_sa has shape (B,)
    q_sa = sum_q_for_action(q_dict, act_t)

    # ------------------------------------------------------------
    # 3) Compute TD target using the TARGET network:
    #      y = r + γ(1-done) * max_a' Q_target(s', a')
    # ------------------------------------------------------------
    with torch.no_grad():
        q_next_dict = qnet_target(next_obs_batch)

        # IMPORTANT NOTE (simplification):
        # We do NOT apply action masks here.
        # Masks are enforced during action selection (acting),
        # but for loss we approximate using unconditional max per branch.
        #
        # If you wanted to be stricter, you'd need to store next-state masks
        # in the replay buffer and apply masked max here.
        q_next_sum = torch.zeros((B,), device=device, dtype=torch.float32)

        # For each branch, take the max over its discrete actions:
        # qk.max(dim=1).values has shape (B,)
        # Then sum across branches to approximate a joint-action max.
        for k, qk in q_next_dict.items():
            q_next_sum += qk.max(dim=1).values

        # TD target vector (B,)
        target = rewards + gamma * (1.0 - dones) * q_next_sum

    # ------------------------------------------------------------
    # 4) Loss: Huber loss (Smooth L1) between current estimate and target.
    # ------------------------------------------------------------
    # Huber is commonly used in DQN because it is less sensitive to outliers
    # than MSE and tends to improve stability.
    loss = F.smooth_l1_loss(q_sa, target)
    return loss


## Safe env.reset() / env.step() wrappers (CivRealm crash recovery)

CivRealm can occasionally crash or throw wrapper errors (server issues, missing masks, KeyErrors).
If we don't handle this, one exception would kill the entire training run.

#### `safe_reset`
- Tries `env.reset(seed=seed)` for reproducibility.
- If the env doesn't accept the `seed` argument, falls back to `env.reset()`.
- If reset crashes, it logs the error + traceback, closes the env, recreates it,
  and resets again so training can continue.

#### `safe_step`
- Tries a normal `env.step(action)`.
- If step crashes, it logs the action and traceback, closes and recreates the env,
  resets to get a valid observation, and returns `terminated=True`.

Returning `terminated=True` is important: after a crash, the next observation does
not correspond to the real successor state of the previous observation, so we end
the episode to avoid poisoning the replay buffer with invalid transitions.


In [ ]:
import time
import traceback
from typing import Callable, Any, Dict


def _close_env_safely(env) -> None:
    """
    Best-effort env.close().

    Why this exists:
    - CivRealm environments can wrap an external game server / subprocess.
    - If something goes wrong, you still want to release ports, processes,
      sockets, file handles, etc.
    - close() itself can sometimes throw (already-dead server), so we swallow
      exceptions to avoid cascading failures.
    """
    try:
        env.close()
    except Exception:
        pass


def safe_reset(
    env,
    make_env_fn: Callable[[], Any],
    seed: int,
    crash_log: list,
):
    """
    Reset the environment safely.

    What it does:
    1) Tries to reset with a seed (for reproducibility).
    2) If the env.reset signature doesn't accept seed (TypeError), falls back
       to env.reset() with no args (compatibility with different wrappers).
    3) If reset fails for ANY reason (server died, wrapper bug, KeyError, etc):
       - logs full traceback + context into crash_log
       - closes the broken env
       - recreates a fresh env using make_env_fn()
       - resets again and returns a valid starting observation

    Why this is needed in CivRealm:
    - The underlying Freeciv server can crash or get stuck.
    - Gym wrappers can raise exceptions when observations/masks are missing.
    - Without recovery, a single crash would kill the whole training run.
    """
    try:
        # Try "modern" gymnasium reset signature first
        try:
            obs, info = env.reset(seed=seed)
        except TypeError:
            # Some envs/wrappers don't accept seed=...
            obs, info = env.reset()
        return env, obs, info

    except Exception as e:
        # Log the crash with enough detail to debug later
        crash_log.append({
            "where": "reset",
            "seed": int(seed),
            "error": repr(e),
            "trace": traceback.format_exc(),
            "time": time.time(),
        })

        # Attempt to cleanly dispose of the broken env and its resources
        _close_env_safely(env)

        # Recreate a fresh environment instance
        env = make_env_fn()

        # Try resetting again (same compatibility logic)
        try:
            obs, info = env.reset(seed=seed)
        except TypeError:
            obs, info = env.reset()

        return env, obs, info


def safe_step(
    env,
    action: Dict[str, int],
    make_env_fn: Callable[[], Any],
    seed: int,
    crash_log: list,
):
    """
    Step the environment safely.

    Normal behavior:
      - env.step(action) returns (next_obs, reward, terminated, truncated, info)

    Recovery behavior (if env.step crashes):
      1) Log the crash (including the action that caused it).
      2) Close the broken env.
      3) Recreate a new env.
      4) Reset to obtain a valid observation (so training code doesn't break).
      5) Force the episode to end cleanly by returning terminated=True.
         This prevents mixing "pre-crash" transitions with a new episode state.

    Why return terminated=True after a crash?
    - A crash breaks the MDP continuity: the next state is not the true
      successor of the current state anymore.
    - Ending the episode avoids adding invalid transitions that would poison
      the replay buffer and destabilize learning.
    """
    try:
        next_obs, reward, terminated, truncated, info = env.step(action)
        return env, next_obs, reward, terminated, truncated, info

    except Exception as e:
        # Log crash details (action is critical context for debugging)
        crash_log.append({
            "where": "step",
            "seed": int(seed),
            "action": {k: int(v) for k, v in action.items()},
            "error": repr(e),
            "trace": traceback.format_exc(),
            "time": time.time(),
        })

        # Dispose the broken env and rebuild from scratch
        _close_env_safely(env)
        env = make_env_fn()

        # Reset to recover a valid observation.
        # We reuse safe_reset because reset can also fail.
        env, obs, info = safe_reset(env, make_env_fn, seed=seed, crash_log=crash_log)

        # Ensure info is a mutable dict (some envs return non-dict info)
        info = dict(info) if isinstance(info, dict) else {"info": info}

        # Tag this transition so later analysis can exclude/inspect it
        info["crash_recovered"] = True
        info["crash_error"] = repr(e)

        # Return a neutral reward (0.0) and terminate the episode.
        # terminated=True tells the training loop "end episode now".
        return env, obs, 0.0, True, False, info


## Training loop

### Training loop: Branching DQN (with replay + target network + safe crash recovery)

This cell implements the full DQN training procedure for CivRealm:

- **Online Q-network (`qnet`)** learns from data.
- **Target Q-network (`qnet_target`)** stabilizes TD targets by being updated only periodically.
- **Replay buffer** stores transitions and provides random mini-batches to reduce correlation.
- **Epsilon-greedy exploration** is decayed linearly to move from exploration → exploitation.
- **Warmup steps** prevent training too early on an undersized, highly correlated buffer.
- **Gradient clipping** reduces instability from rare huge TD errors.
- **safe_reset / safe_step** are critical for CivRealm robustness: if the environment crashes,
  we log the crash, recreate the environment, and end the episode cleanly to avoid corrupt data.


In [ ]:
def train_branching_dqn(
    env,
    total_episodes: int = 50,
    max_steps_per_ep: int = 200,
    device: torch.device = DEVICE,
    gamma: float = 0.99,
    buffer_size: int = 50_000,
    batch_size: int = 64,
    learning_rate: float = 2.5e-4,
    target_update_every: int = 200,
    warmup_steps: int = 500,
    eps_start: float = 1.0,
    eps_end: float = 0.05,
    eps_decay_steps: int = 5_000,
    grad_clip: float = 10.0,
    seed_base: int = 0,
    make_env_fn=make_env_fn,
):
    """
    Train a Branching DQN agent on a CivRealm environment.

    Key ideas implemented here:
    - Two Q networks: online qnet (learns) and target qnet_target (stabilizes targets).
    - Replay buffer: store transitions to break correlation between consecutive samples.
    - Epsilon-greedy exploration with a linear schedule.
    - Warmup period: collect experience before learning to avoid training on tiny buffer.
    - Gradient clipping: reduce instability from occasional large TD errors.
    - Safe reset/step: CivRealm can crash; we log and recover instead of dying mid-run.
    """

    # 1) Create the online Q-network (used for action selection + learning)
    qnet = BranchingQNet(env.observation_space, env.action_space).to(device)

    # 2) Create the target Q-network (used only for bootstrapped TD targets)
    #    We initialize it as a copy of qnet and keep it in eval() mode.
    qnet_target = BranchingQNet(env.observation_space, env.action_space).to(device)
    qnet_target.load_state_dict(qnet.state_dict())
    qnet_target.eval()

    # 3) Optimizer updates the parameters of the online network
    #    Adam is standard for deep RL due to robust adaptive step sizes.
    opt = torch.optim.Adam(qnet.parameters(), lr=learning_rate)

    # 4) Replay buffer stores transitions (s, a, r, s', done)
    #    Sampling random mini-batches helps reduce temporal correlations.
    rb = ReplayBuffer(capacity=buffer_size)

    # 5) Logs for plotting / debugging / comparing runs
    logs = {
        "episode_return": [],  # sum of rewards per episode
        "episode_len": [],     # steps per episode
        "epsilon": [],         # exploration level per episode
        "loss": [],            # TD loss samples during training
        "crashes": [],         # crash records from safe_reset/safe_step
    }

    # Global environment interaction counter (used for epsilon schedule + target updates)
    global_step = 0

    # 6) Initial reset (safe): ensures we start with a valid obs even if env crashes
    env, obs, info = safe_reset(env, make_env_fn, seed=seed_base, crash_log=logs["crashes"])

    # ============================================================
    # Main training loop (episode-by-episode)
    # ============================================================
    for ep in range(total_episodes):
        # Use a deterministic seed per episode (helps reproducibility when possible)
        ep_seed = int(seed_base + ep)

        # 7) Reset at the start of each episode (safe)
        env, obs, info = safe_reset(env, make_env_fn, seed=ep_seed, crash_log=logs["crashes"])

        # Track episode statistics
        ep_ret = 0.0
        ep_len = 0

        # Track how many crashes happened during this episode
        crashes_before = len(logs["crashes"])

        # ------------------------------------------------------------
        # Step loop inside one episode
        # ------------------------------------------------------------
        for t in range(max_steps_per_ep):
            # 8) Epsilon schedule (linear decay)
            #    frac goes from 0 -> 1 as global_step increases.
            #    epsilon moves from eps_start -> eps_end.
            frac = min(1.0, global_step / float(eps_decay_steps))
            epsilon = eps_start + frac * (eps_end - eps_start)

            # 9) Choose an action with epsilon-greedy exploration,
            #    while ALWAYS respecting action masks (CivRealm requirement).
            action = select_action_branching(env, qnet, obs, device, epsilon)

            # 10) Take an environment step (safe):
            #     If env.step crashes, we recover and end the episode cleanly.
            env, next_obs, reward, terminated, truncated, info = safe_step(
                env, action, make_env_fn=make_env_fn, seed=ep_seed, crash_log=logs["crashes"]
            )
            done = bool(terminated or truncated)

            # 11) Store transition in replay buffer.
            #     We strip observations to only the encoder keys (ENC_KEYS)
            #     to keep memory bounded and consistent with the network input.
            rb.push(Transition(
                obs=strip_obs(obs),
                next_obs=strip_obs(next_obs),
                action={k: int(v) for k, v in action.items()},
                reward=float(reward),
                done=done,
            ))

            # 12) Advance to next state and update running counters
            obs = next_obs
            ep_ret += float(reward)
            ep_len += 1
            global_step += 1

            # 13) Learning step (only after buffer is big enough and warmup is done)
            #     Why warmup?
            #     - Early transitions are highly correlated and not diverse.
            #     - Training too early can cause overfitting / divergence.
            if len(rb) >= batch_size and global_step >= warmup_steps:
                batch = rb.sample(batch_size)

                # Put network in training mode (matters if you add dropout/bn later)
                qnet.train()

                # Compute DQN TD loss using the target network for bootstrapping
                loss = compute_branching_dqn_loss(qnet, qnet_target, batch, env, gamma, device)

                # Standard PyTorch training boilerplate
                opt.zero_grad(set_to_none=True)
                loss.backward()

                # 14) Gradient clipping:
                #     helps prevent rare huge TD errors from exploding updates
                nn.utils.clip_grad_norm_(qnet.parameters(), grad_clip)

                opt.step()

                # Log loss for plots (moving averages, stability, etc.)
                logs["loss"].append(float(loss.detach().cpu().item()))

            # 15) Target network update:
            #     periodic hard update is the classic DQN stabilization trick
            if global_step % target_update_every == 0:
                qnet_target.load_state_dict(qnet.state_dict())

            # 16) Episode termination condition:
            #     stop if the environment says terminated/truncated
            if done:
                break

        # 17) Log episode-level metrics (one value per episode)
        logs["episode_return"].append(ep_ret)
        logs["episode_len"].append(ep_len)
        logs["epsilon"].append(float(epsilon))  # last epsilon used in this episode

        # Count crashes that occurred during this episode
        crashes_this_ep = len(logs["crashes"]) - crashes_before

        # 18) Print a concise training line for real-time monitoring
        print(
            f"Ep {ep+1}/{total_episodes} | seed={ep_seed} | return={ep_ret:.2f} | "
            f"len={ep_len} | eps={epsilon:.3f} | rb={len(rb)} | crashes={crashes_this_ep}"
        )

    # Return trained policy network + logs for plotting/analysis
    return qnet, logs


## Training


In [ ]:
qnet, logs = train_branching_dqn(
    env=env,
    total_episodes=300,
    max_steps_per_ep=200,
    device=DEVICE,
)
env.close()
print("Environment closed cleanly")

## Plotting and Analysis

Because of a crash following an 11-hour long training loop, we were unable to retrieve the data from the logs variable. However, we put the printed logs from the training loop in 'dqn_logs_battle.txt' and we will perform the analysis on said logs, which contain all the original info, except for the loss.

In [2]:

LOG_PATH = Path("dqn_logs_battle.txt")


SMOOTH_WINDOW = 20  # moving average window for return/length

# ---- load logs ----
if LOG_PATH.exists():
    text = LOG_PATH.read_text(encoding="utf-8", errors="ignore")
else:
    raise FileNotFoundError(
        f"Couldn't find {LOG_PATH.resolve()} and LOG_TEXT is empty.\n"
        "Either create dqn_logs.txt next to the notebook, set LOG_PATH, or paste logs into LOG_TEXT."
    )

# ---- parse episode lines ----
pat = re.compile(
    r"Ep\s+(?P<ep>\d+)\s*/\s*(?P<total>\d+)\s*\|\s*seed=(?P<seed>-?\d+)\s*\|\s*return=(?P<ret>-?\d+(?:\.\d+)?)\s*\|\s*len=(?P<length>-?\d+)\s*\|\s*eps=(?P<eps>-?\d+(?:\.\d+)?)\s*\|\s*rb=(?P<rb>-?\d+)\s*\|\s*crashes=(?P<crashes>-?\d+)"
)

rows = []
for m in pat.finditer(text):
    d = m.groupdict()
    rows.append({
        "episode": int(d["ep"]),
        "total": int(d["total"]),
        "seed": int(d["seed"]),
        "return": float(d["ret"]),
        "length": int(d["length"]),
        "epsilon": float(d["eps"]),
        "replay_size": int(d["rb"]),
        "crashes": int(d["crashes"]),
    })

if not rows:
    raise ValueError(
        "No episode lines matched.\n"
        "Expected lines like: Ep 1/300 | seed=0 | return=-5.00 | len=139 | eps=0.974 | rb=139 | crashes=0"
    )

df = pd.DataFrame(rows).sort_values("episode").reset_index(drop=True)

def moving_avg(x, w):
    x = np.asarray(x, dtype=float)
    w = max(1, int(w))
    if w == 1 or len(x) < w:
        return np.array([])
    return np.convolve(x, np.ones(w)/w, mode="valid")

# ---- Plot 1: Episode return + moving average ----
plt.figure()
plt.plot(df["episode"], df["return"], marker="o", linestyle="None")
ma = moving_avg(df["return"], SMOOTH_WINDOW)
if len(ma):
    plt.plot(df["episode"].iloc[SMOOTH_WINDOW-1:], ma)
plt.xlabel("Episode")
plt.ylabel("Episode return")
plt.title(f"DQN: Episode return (MA window={SMOOTH_WINDOW})")
plt.grid(True, alpha=0.3)
plt.show()

# ---- Plot 2: Episode length + moving average ----
plt.figure()
plt.plot(df["episode"], df["length"], marker="o", linestyle="None")
ma_len = moving_avg(df["length"], SMOOTH_WINDOW)
if len(ma_len):
    plt.plot(df["episode"].iloc[SMOOTH_WINDOW-1:], ma_len)
plt.xlabel("Episode")
plt.ylabel("Episode length")
plt.title(f"DQN: Episode length (MA window={SMOOTH_WINDOW})")
plt.grid(True, alpha=0.3)
plt.show()

# ---- Plot 3: Epsilon schedule ----
plt.figure()
plt.plot(df["episode"], df["epsilon"])
plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.title("DQN: Epsilon over episodes")
plt.grid(True, alpha=0.3)
plt.show()

# ---- Plot 4: Replay buffer size ----
plt.figure()
plt.plot(df["episode"], df["replay_size"])
plt.xlabel("Episode")
plt.ylabel("Replay buffer size")
plt.title("DQN: Replay buffer size over episodes")
plt.grid(True, alpha=0.3)
plt.show()

# ---- Plot 5: Crashes over episodes ----
plt.figure()
plt.plot(df["episode"], df["crashes"], marker="o", linestyle="None")
plt.xlabel("Episode")
plt.ylabel("Crashes (reported)")
plt.title("DQN: Crashes per episode")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Parsed {len(df)} episodes (max total reported = {int(df['total'].max())}).")
display(df.head(3))
display(df.tail(3))


NameError: name 'Path' is not defined